---
## **0. 준비**
---

In [17]:
%%html
<style>
    body {
        /* change --vscode-font-family if it's different on your system */
        --vscode-font-family: "SeoulNamsanC";
    }
</style>

In [7]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI

In [8]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [1]:
import config
config.set_environment()

---
## **LangChain의 빌딩 블록 살펴보기**
---

### **모델 인터페이스**

<hr style="border:0;border-top:1.5px dashed gray;">

#### **LLM 상호작용 패턴**
- 최근 LangChain 사용 사례를 보면 채팅 모델(ChatModel) 인터페이스를 선호하는 추세 -> 기존 LLM 인터페이스보다 채팅 모델 사용을 권장
- 아래 기본 원리를 이해하면 왜 채팅 모델로의 전환이 필요한지 명확히 알 수 있음

In [ ]:
# 방법 1

# OpenAI 모델 초기화
openai_llm = ChatOpenAI()                                                   #대화형 모델을 호출

# Gemini 모델 초기화
## gemini_pro = GoogleGenerativeAI(model="gemini-1.5-pro")

# 하나 또는 두 모델 모두 같은 인터페이스 사용 가능
response = openai_llm.invoke("전구에 대한 농담 말해줘.")                    #모든 공급자가 동일한 메서드를 사용하는 점이 중요

print(response)

In [12]:
# 방법 2

question = "전구에 대한 농담을 한국어로 하나만 말해줘."

models = [
    "gpt-6-luna",
    "gpt-6.1-sol",
    "gpt-6-astra",
    "gpt-3.5-turbo",    
]

for model_name in models:
    print(f"\n===== {model_name} =====")
    llm = ChatOpenAI(model=model_name, use_responses_api=True,)             #use_responses_api=True 옵션에 대해서는 Craft 문서 참고 

    response = llm.invoke(question)
    #print(response)                                                        #응답 객체 전체: 콘텐츠, 모델 정보, 토큰 사용량 등, 용도는 응답 구조와 부가정보 확인
    #print(response.content)                                                #컨텐츠 원본: 문자열 또는 컨텐츠 블록 목록, 용도는 답변/추론/도구 관련 항목 확인
    print(response.text)                                                    #컨텐츠에서 추출한 답변 텍스트


===== gpt-6-luna =====
전구가 시험을 잘 본 이유는? 늘 밝게 공부했거든요!

===== gpt-6.1-sol =====
프로그래머가 전구를 갈려면 몇 명이 필요할까?

한 명도 필요 없어. “그건 하드웨어 문제잖아요.” 💡

===== gpt-6-astra =====
전구 하나 갈아 끼우는 데 프로그래머가 몇 명 필요할까?

한 명도 필요 없어. 다들 “그건 하드웨어 문제잖아!”라고 하거든.

===== gpt-3.5-turbo =====
"전구가 인사를 할 때는 어떻게 하나요? 전굴 전굴!"


<hr style="border:0;border-top:1.5px dashed gray;">

#### **개발 테스트**
- 개발 단계에서는 실제 API 호출 없이도 애플리케이션을 테스트 할 수 있음
- LangChain은 이러한 목적으로 `FakeListLLM`을 제공

In [3]:
from langchain_community.llms import FakeListLLM

# 항상 동일한 응답을 반환하는 가짜 LLM 생성
fake_llm = FakeListLLM(responses=["이것은 테스트 응답입니다."])

result = fake_llm.invoke("어떤 인풋도 위 응답으로 도출")

print(result)

이것은 테스트 응답입니다.


/tmp/ipykernel_7389/853906123.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import FakeListLLM


<hr style="border:0;border-top:1.5px dashed gray;">

#### **채팅 모델과 작업**
- 요즘 모델 공급자가 제공하는 API는 각 대화 턴을 별도의 잘 정리된 페이로드(payload) 형태로 처리하도록 되어 있음.
- 모델 공급자들은 일반적으로 채팅 기록을 서버 측에 저장하지 않으며, 매번 클라이언트로부터 전체 기록을 전송받은 후 서버에서 최종 프롬프트만 구조화하여 처리.
- LangChain은 채팅 모델에서도 이와 같은 패턴을 따르며, 역할과 내용이 명시된 구조화된 대화를 처리. 각 메시지는 다음 요소를 포함
    - 역할(누가 말하는지): 메세지 클래스로 정의됨(모든 메세지는 `BaseMessage`를 상속)
    - 내용(무엇을 말하는지)
- **메세지 유형**
    - **SystemMessage** : 모델의 기본 동작 방식 및 역할을 설정.
        - `SystemMessage(content= "You're a helpful programming assistant")`
    - **HumanMessage** : 질문, 명령어, 데이터 등의 사용자 입력.
        - `HumanMessage(content= "Factorial을 계산하는 함수를 작성해줘")`
    - **AIMessage** : 모델의 응답을 포함


In [16]:
chat = ChatOpenAI(model = "gpt-6-luna", use_responses_api=True,)
messages = [
    SystemMessage(content="You're a helpful programming assistant"),
    HumanMessage(content="Factorial을 계산하는 파이썬 함수를 작성해줘. 설명 및 사용 예시도 함께 제공해야돼"),
]
response = chat.invoke(messages)
print(response.text)

```python
def factorial(n):
    """0 이상의 정수 n의 팩토리얼을 계산합니다."""
    if type(n) is not int:
        raise TypeError("n은 정수여야 합니다.")
    if n < 0:
        raise ValueError("n은 0 이상이어야 합니다.")

    result = 1
    for number in range(2, n + 1):
        result *= number

    return result
```

팩토리얼은 `n!`으로 나타내며, `1`부터 `n`까지의 모든 정수를 곱한 값입니다. 예를 들어 `5! = 1 × 2 × 3 × 4 × 5 = 120`이고, `0!`은 `1`입니다.

사용 예시:

```python
print(factorial(5))  # 120
print(factorial(0))  # 1
```

정수가 아닌 값을 입력하면 `TypeError`, 음수를 입력하면 `ValueError`가 발생합니다.


<hr style="border:0;border-top:1.5px dashed gray;">

#### **추론 모델**

2026년 10월 13일 기준 변화는 다음과 같습니다.

| 구분 | 교재의 방식 | 최신 방식 |
|---|---|---|
| Claude | `budget_tokens`로 사고 토큰 예산 지정 | **최신 지원 모델은 적응형 사고와 `effort` 사용** |
| GPT | o3의 `reasoning_effort` | **최신 GPT에서도 추론 강도를 설정하며, 지원 단계는 모델별로 다름** |

Claude는 4.6에서 수동 예산 방식이 deprecated되었고, 4.7 이후 모델에서는 지원하지 않습니다. 따라서 “Claude는 토큰 예산, GPT는 강도”라는 교재의 구분이 최신 모델에는 그대로 적용되지 않습니다. [Claude 공식 문서](https://platform.claude.com/docs/en/build-with-claude/extended-thinking)

GPT 실습에는 `gpt-6-luna`를 사용했습니다. 이 모델은 `none`, `low`, `medium`, `high`, `xhigh`, `max`를 지원합니다. **강도는 정확한 토큰 수를 지정하는 값이 아닙니다.** [OpenAI 공식 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)

추가한 셀의 핵심 설정은 다음과 같습니다.

```python
reasoning={"effort": effort, "summary": "auto"}
```

`reasoning_effort`와 같은 강도 설정을 딕셔너리에 넣고, 추론 요약도 함께 요청했습니다. **요약은 내부 사고 과정의 원문이 아니며, 응답에 없을 수도 있습니다.** [추론 안내](https://developers.openai.com/api/docs/guides/reasoning)

셀을 실행하면 동일한 다리 건너기 문제를 `low`와 `high`로 풀고 다음을 비교합니다.

- 최종 답변과 추론 요약
- 추론 토큰·전체 출력 토큰 사용량
- 응답 시간과 검산 기준인 17분

코드 구문과 실제 요청 설정 변환을 검증했습니다. **API 호출은 실행하지 않았으며**, 사용자가 셀을 실행하면 총 2회 호출되어 비용이 발생합니다.

In [19]:
# GPT 추론 강도 비교 실습 (2026-10-13 공식 문서 기준)
# 실행하면 같은 질문으로 API를 2번 호출합니다. 추론 토큰도 출력 요금에 포함됩니다.
# Claude 최신 지원 모델: thinking={"type": "adaptive"}, output_config={"effort": "high"}
# GPT: reasoning_effort로 강도를 지정. 토큰 수를 지정하는 budget_tokens와는 다릅니다.
# GPT-6 Luna 지원 강도: none, low, medium(기본), high, xhigh, max

from time import perf_counter                                               #경과 시간을 정밀하게 측정하는 함수

# 기본 설정
MODEL = "gpt-6-luna"                                                        #비교할 모델 ID를 지정. 강도 비교에서는 모델을 동일하게 유지
EFFORTS = ["low", "high"]                                                   #반복문에서 low, high 순서로 실행할 추론 강도를 목록에 저장.
MAX_OUTPUT_TOKENS = 25_000                                                  #내부 추론 + 최종 답변의 합계 상한 (사고 예산이 아님)

# 여러 줄 문자열로 동일한 질문을 저장합니다. 따옴표 안의 각 줄은 모두 모델에게 보낼 문제 조건입니다.
question = """네 사람이 밤에 다리를 건너야 한다. 각자 다리를 건너는 데 1분, 2분, 5분, 10분이 걸린다.
다리에는 최대 두 사람만 동시에 올라갈 수 있다. 두 사람이 함께 건너면 느린 사람의 시간이 걸린다.
손전등은 하나이며, 건널 때마다 반드시 가지고 있어야 한다. 모두 처음에는 왼쪽에 있고, 손전등을 가지고 돌아오는 시간도 포함한다.
모두 오른쪽으로 이동하는 최소 총 시간과 이동 순서를 구하라. 한국어로 최종 답과 검증 가능한 간결한 근거를 제시하라."""

reasoning_results = {}                                                      #빈 딕셔너리 생성, 실행 후 reasoning_results["low"] 처럼 응답 재확인 가능, 재호출 없이 response/content/text를 살펴볼 수 있음

# 가동
for effort in EFFORTS:
    print(f"\n{'=' * 20} {MODEL} / effort={effort} {'=' * 20}")
    
    # 이번 반복에서 사용할 모델 객체를 만듭니다. 아직 API를 호출하지는 않습니다.
    llm = ChatOpenAI(
        model=MODEL,                                                        #위에서 지정한 모델 ID를 전달
        use_responses_api=True,                                             #추론 요약을 지원하는 Responses API로 요청하도록 지정
                
        reasoning={"effort": effort, "summary": "auto"},                    #원본 사고가 아닌 요약 요청, reasoning_effort=effort와 같은 강도 설정을 reasoning 딕셔너리에 지정, 이 환경에서는 두 설정을 함께 쓰면 딕셔너리가 우선하므로 한곳에 모음
                                                                            #effort로 추론 강도를, summary="auto"로 제공 가능한 추론 요약을 요청. 정확한 추론 토큰 수는 지정하지 않음.
        max_tokens=MAX_OUTPUT_TOKENS,                                       #Responses API의 max_output_tokens로 변환, 내부 추론과 최종 답변을 합친 출력 상한. 너무 작으면 최종 답변 전에 토큰이 소진될 수 있음.
        timeout=180,                                                        #요청 대기 제한을 180초로 지정
        max_retries=0,                                                      #요청 실패 시 LangChain/OpenAI 클라이언트의 자동 재시도를 하지 않음
    )

    # 추론-응답 시간 측정
    started = perf_counter()                                                #API 요청 직전의 시각을 기록, 달력 시각이 아니라 경과 시간 측정용 값.
    response = llm.invoke(question)                                         #실제 API를 호출하고 응답 객체를 저장, 이 줄에서 사용량에 따른 비용이 발생.
    elapsed = perf_counter() - started                                      #응답을 받은 시각에서 시작 시각을 빼 전체 대기 시간을 초 단위로 계산.
    
    # 응답 저장 및 출력
    reasoning_results[effort] = response                                    #현재 강도를 키로 응답을 보관, 다음 반복이 response를 덮어써도 이전 응답이 남음.

    print("\n[최종 답변]")
    print(response.text or "답변 텍스트 없음: 토큰 상한과 응답 상태를 확인하세요.") #답변 텍스트를 출력, 빈 문자열이면 or 오른쪽의 안내 문구를 출력.

    # 추론 요약 출력 구간 시작, 실제 내부 사고 원문은 공개되지 않음.
    print("\n[추론 요약 — 내부 사고 원문이 아닙니다]")                     
        # 조건에 맞는 요약만 모으는 리스트 컴프리헨션을 시작합니다.
        # [결과로 넣을 값 for 변수 in 반복할 대상 if 조건]
    summaries = [
        block["reasoning"]                                                  #조건을 통과한 콘텐츠 블록의 reasoning 문자열을 결과 목록에 넣음
        for block in response.content_blocks                                #응답을 표준화한 콘텐츠 블록 목록에서 블록을 하나씩 읽음
        if block.get("type") == "reasoning" and block.get("reasoning")      #type이 reasoning이고 요약 내용이 비어 있지 않은 블록만 선택, get은 키가 없어도 오류 대신 None을 반환.
    ]
    
    # 요약들을 줄바꿈으로 연결, 빈 목록이면 요약이 없다는 안내를 출력.
    print("\n".join(summaries) if summaries else "이 응답에는 공개된 추론 요약이 없습니다.")

    # 토큰 사용량 딕셔너리를 가져옴, 정보가 None이면 빈 딕셔너리로 대체.
    usage = response.usage_metadata or {}
    
    # 출력 토큰의 세부 정보에서 추론 토큰 수를 읽음, 세부 정보가 없으면 빈 딕셔너리를 사용합니다.
    reasoning_tokens = usage.get("output_token_details", {}).get("reasoning")
    
    # 입력 토큰 수를 문자열에 넣음, 해당 키가 없으면 미제공으로 표시.
    print(f"\n[사용량] 입력={usage.get('input_tokens', '미제공')}, "
          # 추론 토큰도 포함하는 출력 토큰 수를 앞 문자열에 이어 붙입니다.
          f"출력(추론 포함)={usage.get('output_tokens', '미제공')}, "
          # 추론 토큰 수가 None이면 미제공을 표시합니다. 0은 유효한 사용량이므로 그대로 표시합니다.
          f"추론={reasoning_tokens if reasoning_tokens is not None else '미제공'}")
    
    # 측정한 초를 소수점 둘째 자리까지 표시, 네트워크 대기 시간도 포함.
    print(f"[응답 시간] {elapsed:.2f}초")
    # 응답 메타데이터의 상태를 표시, status 항목이 없으면 미제공을 표시.
    print("[응답 상태]", response.response_metadata.get("status", "미제공"))

# 두 응답을 확인할 정답과 이동 순서를 출력.
print("\n[검산 기준] 최소 17분: 1·2 건너기(2) → 1 돌아오기(1) → 5·10 건너기(10) → 2 돌아오기(2) → 1·2 건너기(2).")
print("한 번의 결과만으로 성능을 판단하지 마세요. 높은 강도가 항상 더 느리거나 정확한 것은 아닙니다.")



==================== gpt-6-luna / effort=low ====================



[최종 답변]
**최소 시간은 17분**입니다.

| 순서 | 이동 | 걸린 시간 |
|---|---|---:|
| 1 | 1분·2분 사람이 오른쪽으로 건넌다 | 2분 |
| 2 | 1분 사람이 왼쪽으로 돌아온다 | 1분 |
| 3 | 5분·10분 사람이 오른쪽으로 건넌다 | 10분 |
| 4 | 2분 사람이 왼쪽으로 돌아온다 | 2분 |
| 5 | 1분·2분 사람이 오른쪽으로 건넌다 | 2분 |
| **합계** |  | **17분** |

각 이동 때 손전등을 함께 옮기며, 마지막에는 네 사람 모두 오른쪽에 있습니다.

**최소임을 확인하면:** 가장 느린 두 사람(5분·10분)이 건너는 데는 두 가지 유리한 방식이 있습니다. 빠른 두 사람이 함께 셔틀 역할을 하면 `2+1+10+2+2=17분`이고, 가장 빠른 사람이 느린 두 사람을 따로 안내하면 `10+1+5+1+2=19분`입니다. 어느 방식도 17분보다 짧지 않으므로, 위 순서가 최적입니다.

[추론 요약 — 내부 사고 원문이 아닙니다]
이 응답에는 공개된 추론 요약이 없습니다.

[사용량] 입력=150, 출력(추론 포함)=808, 추론=516
[응답 시간] 11.28초
[응답 상태] completed

==================== gpt-6-luna / effort=high ====================

[최종 답변]
최소 총 시간은 **17분**입니다.

| 순서 | 이동 | 시간 |
|---|---|---:|
| 1 | 1분·2분 사람이 오른쪽으로 건넘 | 2분 |
| 2 | 1분 사람이 손전등을 들고 돌아옴 | 1분 |
| 3 | 5분·10분 사람이 오른쪽으로 건넘 | 10분 |
| 4 | 2분 사람이 손전등을 들고 돌아옴 | 2분 |
| 5 | 1분·2분 사람이 오른쪽으로 건넘 | 2분 |
| **합계** |  | **17분** |

**최소임을 확인하면:** 가장 느린 두 사람(5분, 10분)을 함께 보내는 방식의 최소 시간은  
\(1+3\times2+10=17\)분입니다

<hr style="border:0;border-top:1.5px dashed gray;">

#### **모델 동작 제어**
